In [ ]:
%pip install -q groq ollama ipywidgets python-dotenv

In [ ]:
import os, json, re, unicodedata
from collections import deque
from itertools import product

PROVEDOR = "groq" # "groq" (nuvem), "ollama" (local) ou "offline" (sem LLM)
MODELO_GROQ = "openai/gpt-oss-20b" # confira em console.groq.com/docs/models
MODELO_OLLAMA = "llama3.2" # baixado com: ollama pull llama3.2

def obter_chave_groq():

    try:
        from google.colab import userdata
        return userdata.get("GROQ_API_KEY")
    except Exception:
        pass
    try:
       from dotenv import load_dotenv
       load_dotenv()
    except Exception:
       pass
    return os.environ.get("GROQ_API_KEY")

def chamar_llm(mensagens, modo_json=False):

    if PROVEDOR == "groq":
      from groq import Groq
      cliente = Groq(api_key=obter_chave_groq())
      extras = {"response_format": {"type": "json_object"}} if modo_json else {}
      resposta = cliente.chat.completions.create(
          model=MODELO_GROQ, messages=mensagens, temperature=0, **extras)

      return resposta.choices[0].message.content
    elif PROVEDOR == "ollama":
      import ollama
      extras = {"format": "json"} if modo_json else {}
      resposta = ollama.chat(model=MODELO_OLLAMA, messages=mensagens,
      options={"temperature": 0}, **extras)
      return resposta["message"]["content"]
    else:
        raise RuntimeError("Modo offline: nenhum LLM configurado.")




In [ ]:
resposta = chamar_llm([
    {"role": "user",
      "content": "Em uma frase curta, dê boas-vindas aos passageiros do metrô de São Paulo."}
])
print(resposta)

Bem-vindos ao metrô de São Paulo!


In [ ]:
LINHA_1_AZUL = [
"Tucuruvi", "Parada Inglesa", "Jardim São Paulo", "Santana",
"Carandiru", "Portuguesa-Tietê", "Armênia", "Tiradentes", "Luz",
"São Bento", "Sé", "Japão-Liberdade", "São Joaquim", "Vergueiro",
"Paraíso", "Ana Rosa", "Vila Mariana", "Santa Cruz",
"Praça da Árvore", "Saúde", "São Judas", "Conceição", "Jabaquara",
]
def construir_grafo(estacoes):
    """Cada estação vira um nó ligado à anterior e à próxima da lista."""
    grafo = {estacao: [] for estacao in estacoes}
    for i in range(len(estacoes) - 1):
        a, b = estacoes[i], estacoes[i + 1]
        grafo[a].append(b) # a → b
        grafo[b].append(a) # b → a (o trem anda nos dois sentidos)
    return grafo
GRAFO = construir_grafo(LINHA_1_AZUL)

In [ ]:
# Célula 5 — explorando o grafo
print("Total de estações:", len(GRAFO))
print("Vizinhas da Sé:", GRAFO["Sé"])
print("Vizinhas do Tucuruvi:", GRAFO["Tucuruvi"])

for estacao, vizinhas in list(GRAFO.items())[:5]:
    print(f"{estacao:>18} → {vizinhas}")

Total de estações: 23
Vizinhas da Sé: ['São Bento', 'Japão-Liberdade']
Vizinhas do Tucuruvi: ['Parada Inglesa']
          Tucuruvi → ['Parada Inglesa']
    Parada Inglesa → ['Tucuruvi', 'Jardim São Paulo']
  Jardim São Paulo → ['Parada Inglesa', 'Santana']
           Santana → ['Jardim São Paulo', 'Carandiru']
         Carandiru → ['Santana', 'Portuguesa-Tietê']


In [ ]:
LOCAIS = {
"Shopping Metrô Tucuruvi": "Tucuruvi",
"Terminal Rodoviário Tietê": "Portuguesa-Tietê",
"Museu de Arte Sacra": "Tiradentes",
"Pinacoteca": "Luz",
"Museu da Língua Portuguesa": "Luz",
"Mosteiro de São Bento": "São Bento",
"Rua 25 de Março": "São Bento",
"Catedral da Sé": "Sé",
"Bairro da Liberdade": "Japão-Liberdade",
"Centro Cultural São Paulo": "Vergueiro",
"Shopping Metrô Santa Cruz": "Santa Cruz",
"Universidade São Judas": "São Judas",
"Terminal Rodoviário Jabaquara": "Jabaquara",
}

In [ ]:
# Célula 7 — BFS passo a passo
def bfs_passo_a_passo(grafo, origem, destino):
    fila = deque([origem])
    pai = {origem: None}
    passo = 0
    while fila:
        passo += 1
        atual = fila.popleft()
        if atual == destino:
            print(f"Passo {passo}: visito {atual:<17} Cheguei!")
            return pai # devolve o "fio de Ariadne" (próximo passo!)
        for vizinho in grafo[atual]:
          if vizinho not in pai:
              pai[vizinho] = atual
              fila.append(vizinho)
    print(f"Passo {passo}: visito {atual:<17} | fila: {list(fila)}")
    return pai
pai = bfs_passo_a_passo(GRAFO, "Sé", "São Joaquim")
print("\nDicionário pai:", pai)



Passo 5: visito São Joaquim       Cheguei!

Dicionário pai: {'Sé': None, 'São Bento': 'Sé', 'Japão-Liberdade': 'Sé', 'Luz': 'São Bento', 'São Joaquim': 'Japão-Liberdade', 'Tiradentes': 'Luz'}


In [ ]:
def reconstruir_caminho(pai, destino):
  """Puxa o 'fio de Ariadne': do destino até a origem, depois inverte."""
  caminho = []
  atual = destino
  while atual is not None:
      caminho.append(atual)
      atual = pai[atual]
  return list(reversed(caminho))


def bfs(grafo, origem, destino, bloqueadas=()):
    if origem in bloqueadas or destino in bloqueadas:
        return None, []
    fila = deque([origem])
    pai = {origem: None}
    ordem_visita = []
    while fila:
        atual = fila.popleft() # 1º da fila sai primeiro (FIFO)
        ordem_visita.append(atual)
        if atual == destino:
            return reconstruir_caminho(pai, destino), ordem_visita
        for vizinho in grafo[atual]:
            if vizinho not in pai and vizinho not in bloqueadas:
              pai[vizinho] = atual
              fila.append(vizinho)
    return None, ordem_visita

In [ ]:
caminho, visitados = bfs(GRAFO, "Luz", "Vergueiro")
print("Caminho:", " → ".join(caminho))
print("Paradas:", len(caminho) - 1)
print("Estações visitadas pela busca:", len(visitados))

# E se Sé estiver fechada?
caminho, visitados = bfs(GRAFO, "Luz", "Vergueiro", bloqueadas={"Sé"})
print("\nCom a Sé fechada:", caminho)


Caminho: Luz → São Bento → Sé → Japão-Liberdade → São Joaquim → Vergueiro
Paradas: 5
Estações visitadas pela busca: 11

Com a Sé fechada: None


In [ ]:
def dfs(grafo, origem, destino, bloqueadas=()):

    if origem in bloqueadas or destino in bloqueadas:
        return None, []
    visitados = set()
    ordem_visita = []

    def explorar(atual, caminho):
        visitados.add(atual)
        ordem_visita.append(atual)
        if atual == destino:
           return caminho
        for vizinho in grafo[atual]:
            if vizinho not in visitados and vizinho not in bloqueadas:
                resultado = explorar(vizinho, caminho + [vizinho]) # mergulha
                if resultado:
                  return resultado
    return None # beco sem saída → volta
    return explorar(origem, [origem]), ordem_visita


In [ ]:
def dfs(grafo, origem, destino, bloqueadas=()):

    if origem in bloqueadas or destino in bloqueadas:
        return None, []
    visitados = set()
    ordem_visita = []

    def explorar(atual, caminho):
        visitados.add(atual)
        ordem_visita.append(atual)
        if atual == destino:
           return caminho
        for vizinho in grafo[atual]:
            if vizinho not in visitados and vizinho not in bloqueadas:
                resultado = explorar(vizinho, caminho + [vizinho]) # mergulha
                if resultado:
                  return resultado
        return None # beco sem saída → volta (agora indentado corretamente dentro de explorar)

    return explorar(origem, [origem]), ordem_visita

In [ ]:
viagens = [("Sé", "Japão-Liberdade"), ("Sé", "São Joaquim"),
("Luz", "Vergueiro"), ("Santana", "Sé"), ("Paraíso", "Tucuruvi")]


print(f"{'Viagem':<28}{'Paradas':>8}{'BFS visitou':>13}{'DFS visitou':>13}")
for origem, destino in viagens:
    c_bfs, v_bfs = bfs(GRAFO, origem, destino)
    c_dfs, v_dfs = dfs(GRAFO, origem, destino)
    print(f"{origem + ' → ' + destino:<28}{len(c_bfs) - 1:>8}{len(v_bfs):>13}{len(v_dfs):>13}")

_, v = dfs(GRAFO, "Sé", "Japão-Liberdade")
print("\nOrdem da DFS de Sé até Japão-Liberdade:", v)


Viagem                       Paradas  BFS visitou  DFS visitou
Sé → Japão-Liberdade               1            3           12
Sé → São Joaquim                   2            5           13
Luz → Vergueiro                    5           11           14
Santana → Sé                       7           11           11
Paraíso → Tucuruvi                14           23           15

Ordem da DFS de Sé até Japão-Liberdade: ['Sé', 'São Bento', 'Luz', 'Tiradentes', 'Armênia', 'Portuguesa-Tietê', 'Carandiru', 'Santana', 'Jardim São Paulo', 'Parada Inglesa', 'Tucuruvi', 'Japão-Liberdade']


In [ ]:
def pode_embarcar(P, Q, R):
    """P: estação aberta | Q: precisa de acessibilidade | R: elevador funcionando"""
    return P and ((not Q) or R)
def tabela_verdade():
    print(" P | Q | R | P ∧ (¬Q ∨ R)")
    print("-" * 40)
    for P, Q, R in product([True, False], repeat=3):
        print(f" {P!s:5} | {Q!s:5} | {R!s:5} | {pode_embarcar(P, Q, R)}")
tabela_verdade()


 P | Q | R | P ∧ (¬Q ∨ R)
----------------------------------------
 True  | True  | True  | True
 True  | True  | False | False
 True  | False | True  | True
 True  | False | False | True
 False | True  | True  | False
 False | True  | False | False
 False | False | True  | False
 False | False | False | False


In [ ]:
def fatos_base():
    """Fatos fixos do mundo: quais estações existem e o que fica perto de cada uma."""
    fatos = set()
    for estacao in LINHA_1_AZUL:
        fatos.add(("estacao", estacao))
    for local, estacao in LOCAIS.items():
        fatos.add(("proximo_de", local, estacao))
    return fatos

def consultar(fatos, predicado):
    """Devolve os argumentos de todos os fatos de um predicado. Ex.: consultar(f, 'destino') → [('Luz',)]"""
    return [f[1:] for f in fatos if f[0] == predicado]
def r_origem(fatos):
    novos = set()
    for (local,) in consultar(fatos, "usuario_esta_em"):
        for (l, e) in consultar(fatos, "proximo_de"):
          if l == local:
              novos.add(("origem", e))
    for (e,) in consultar(fatos, "usuario_esta_na_estacao"):
        novos.add(("origem", e))
    return novos

def r_destino(fatos):
    novos = set()
    for (local,) in consultar(fatos, "usuario_quer_ir"):
      for (l, e) in consultar(fatos, "proximo_de"):
        if l == local:
          novos.add(("destino", e))
    for (e,) in consultar(fatos, "usuario_quer_ir_estacao"):
        novos.add(("destino", e))
    return novos

def r_bloqueio(fatos):
    return {("bloqueada", e) for (e,) in consultar(fatos, "fechada")}

def r_acessibilidade(fatos):
    if not consultar(fatos, "precisa_acessibilidade"):
      return set()
    return {("inacessivel", e) for (e,) in consultar(fatos, "elevador_em_manutencao")}

def r_alerta(fatos):
    novos = set()
    inacessiveis = {e for (e,) in consultar(fatos, "inacessivel")}
    for papel in ("origem", "destino"):
       for (e,) in consultar(fatos, papel):
         if e in inacessiveis:
           novos.add(("alerta", papel, e))
    return novos

REGRAS = [
("R1 origem", "∀l ∀e (usuario_esta_em(l) ∧ proximo_de(l,e) → origem(e))", r_origem),
("R2 destino", "∀l ∀e (usuario_quer_ir(l) ∧ proximo_de(l,e) → destino(e))", r_destino),
("R3 bloqueio", "∀e (fechada(e) → bloqueada(e))", r_bloqueio),
("R4 acessibilidade","∀e (precisa_acessibilidade ∧ elevador_em_manutencao(e) → inacessivel(e))", r_acessibilidade),
("R5 alerta", "∀p ∀e (papel(p,e) ∧ inacessivel(e) → alerta(p,e))", r_alerta),
]
def encadear_para_frente(fatos, regras, verbose=False):
    """Aplica as regras em rodadas até não surgir nenhum fato novo (ponto fixo)."""
    fatos = set(fatos)
    justificativas = {}
    rodada = 0
    while True:
        rodada += 1
        novos_na_rodada = set()
        for nome, _formula, regra in regras:
          for fato in regra(fatos) - fatos:
              novos_na_rodada.add(fato)
              justificativas[fato] = nome
        if verbose:
            print(f"Rodada {rodada}: {len(novos_na_rodada)} fato(s) novo(s)")
        if not novos_na_rodada:
            return fatos, justificativas
        fatos |= novos_na_rodada




In [ ]:
fatos = fatos_base() | {
    ("usuario_quer_ir", "Pinacoteca"),
    ("precisa_acessibilidade",),
    ("elevador_em_manutencao", "Luz"),
}

fatos, justificativas = encadear_para_frente(fatos, REGRAS, verbose=True)
print("\nFatos deduzidos e suas justificativas:")
for fato, regra in sorted(justificativas.items(), key=lambda x: x[1]):
    print(f" {fato} ⟵ {regra}")

print("\nConsulta: qual é o destino?", consultar(fatos, "destino"))

Rodada 1: 2 fato(s) novo(s)
Rodada 2: 1 fato(s) novo(s)
Rodada 3: 0 fato(s) novo(s)

Fatos deduzidos e suas justificativas:
 ('destino', 'Luz') ⟵ R2 destino
 ('inacessivel', 'Luz') ⟵ R4 acessibilidade
 ('alerta', 'destino', 'Luz') ⟵ R5 alerta

Consulta: qual é o destino? [('Luz',)]


In [ ]:
TEMPO_POR_TRECHO = 2 # minutos por trecho (valor simulado, didático)

def planejar(pedido, fechadas=(), manutencao=(), algoritmo="BFS"):
    """pedido = {"origem": (tipo, nome), "destino": (tipo, nome), "acessibilidade": bool}
    tipo é "local" ou "estacao"."""
    # 1) Monta a base de conhecimento com o pedido e o cenário
    fatos = fatos_base()
    tipo_o, nome_o = pedido["origem"]
    tipo_d, nome_d = pedido["destino"]
    fatos.add(("usuario_esta_em", nome_o) if tipo_o == "local" else ("usuario_esta_na_estacao", nome_o))
    fatos.add(("usuario_quer_ir", nome_d) if tipo_d == "local" else ("usuario_quer_ir_estacao", nome_d))
    if pedido.get("acessibilidade"):
        fatos.add(("precisa_acessibilidade",))
    for e in fechadas:
        fatos.add(("fechada", e))
    for e in manutencao:
        fatos.add(("elevador_em_manutencao", e))

    # 2) Inferência lógica
    fatos, justificativas = encadear_para_frente(fatos, REGRAS)
    origem = consultar(fatos, "origem")[0][0]
    destino = consultar(fatos, "destino")[0][0]
    bloqueadas = {e for (e,) in consultar(fatos, "bloqueada")}
    alertas = consultar(fatos, "alerta")
    # 3) Busca usando o que a lógica deduziu
    buscar = bfs if algoritmo == "BFS" else dfs
    caminho, visitados = buscar(GRAFO, origem, destino, bloqueadas)
    return {
        "origem": origem, "destino": destino, "algoritmo": algoritmo,
        "caminho": caminho, "visitados": visitados,
        "bloqueadas": sorted(bloqueadas),
        "alertas": [f"{papel}: {e}" for papel, e in alertas],
        "paradas": len(caminho) - 1 if caminho else None,
        "tempo_min": (len(caminho) - 1) * TEMPO_POR_TRECHO if caminho else None,
        "regras_usadas": sorted(set(justificativas.values())),
        }


In [ ]:
r = planejar({"origem": ("local", "Catedral da Sé"),
            "destino": ("local", "Pinacoteca"),
            "acessibilidade": True},
            manutencao=["Luz"])

for chave, valor in r.items():
    print(f"{chave:>14}: {valor}")


        origem: Sé
       destino: Luz
     algoritmo: BFS
       caminho: ['Sé', 'São Bento', 'Luz']
     visitados: ['Sé', 'São Bento', 'Japão-Liberdade', 'Luz']
    bloqueadas: []
       alertas: ['destino: Luz']
       paradas: 2
     tempo_min: 4
 regras_usadas: ['R1 origem', 'R2 destino', 'R4 acessibilidade', 'R5 alerta']


In [ ]:
def normalizar(texto):
    """Minúsculas e sem acentos: 'São Bento' → 'sao bento'."""
    texto = unicodedata.normalize("NFD", texto.lower())
    return "".join(c for c in texto if unicodedata.category(c) != "Mn")

def resolver_nome(nome):
    """GUARDRAIL: só aceita nomes que existem de verdade. Senão, None."""
    if not nome:
        return None
    alvo = normalizar(nome).strip()
    for estacao in LINHA_1_AZUL:
        if normalizar(estacao) == alvo:
          return ("estacao", estacao)
    for local in LOCAIS:
        if normalizar(local) == alvo:
           return ("local", local)
    return None

PROMPT_INTERPRETE = """Você é o módulo de INTERPRETAÇÃO do MetrôBot SP.
Sua única tarefa é transformar o pedido do passageiro em JSON.
Estações válidas: {estacoes}
Locais válidos: {locais}
Responda APENAS com um JSON neste formato:
{{"origem": "<nome exato de estação ou local, ou null>",
"destino": "<nome exato de estação ou local, ou null>",
"acessibilidade": <true ou false>}}
Regras:
- Use SOMENTE nomes das listas acima, escritos exatamente como aparecem.
- "acessibilidade" é true se o passageiro mencionar cadeira de rodas,
mobilidade reduzida, muletas, carrinho de bebê ou precisar de elevador.
"""

def interpretar_offline(texto):
    """Plano B sem LLM: procura nomes conhecidos no texto, na ordem em que aparecem."""
    texto_min = texto.lower()
    texto_sem = normalizar(texto) # mesmo tamanho, só sem acentos
    candidatos = [(n, "estacao") for n in LINHA_1_AZUL] + [(n, "local") for n in LOCAIS]
    candidatos.sort(key=lambda c: len(c[0]), reverse=True) # nomes longos primeiro
    ocupado = [False] * len(texto_min)
    encontrados = []
    for nome, tipo in candidatos:
        buscas = [(texto_min, nome.lower())]
        if len(nome) > 4 and len(texto_sem) == len(texto_min):
            buscas.append((texto_sem, normalizar(nome))) # aceita "se" sem acento só p/ nomes longos
        for base, padrao in buscas:
          for m in re.finditer(r"(?<!\w)" + re.escape(padrao) + r"(?!\w)", base):
             if not any(ocupado[m.start():m.end()]):
              encontrados.append((m.start(), nome))
              for i in range(m.start(), m.end()):
                ocupado[i] = True
    encontrados.sort()
    palavras_acess = ["cadeira de rodas", "acessibilidade", "mobilidade",
    "muleta", "carrinho de bebe", "elevador"]
    return {
    "origem": encontrados[0][1] if len(encontrados) > 0 else None,
    "destino": encontrados[1][1] if len(encontrados) > 1 else None,
    "acessibilidade": any(p in texto_sem for p in palavras_acess),
    }
def interpretar_pedido(texto):
    """Texto livre → pedido validado. Usa o Llama; se falhar, cai no modo offline."""
    if PROVEDOR == "offline":
      bruto = interpretar_offline(texto)
      fonte = "offline"
    else:
        sistema = PROMPT_INTERPRETE.format(
          estacoes=", ".join(LINHA_1_AZUL), locais=", ".join(LOCAIS))
        try:
            resposta = chamar_llm([{"role": "system", "content": sistema},
                                    {"role": "user", "content": texto}], modo_json=True)
            bruto = json.loads(resposta)
            fonte = PROVEDOR

        except Exception as erro:
            print(f" LLM indisponível ({erro}). Usando modo offline.")
            bruto = interpretar_offline(texto)
            fonte = "offline"
    origem = resolver_nome(bruto.get("origem"))
    destino = resolver_nome(bruto.get("destino"))
    if origem is None or destino is None:
         return None, f"Não entendi origem/destino (resposta bruta: {bruto})"
    pedido = {"origem": origem, "destino": destino,
    "acessibilidade": bool(bruto.get("acessibilidade"))}
    return pedido, f"Interpretado via {fonte}"


In [ ]:
pedidos = [
"Estou na Catedral da Sé e quero ir ao Terminal Rodoviário Jabaquara",
"to na se, bora pra pinacoteca, tô de cadeira de rodas",
"Preciso sair do Mosteiro de São Bento e chegar na São Judas",
"Quero ir da Sé até a Avenida Paulista",
]
for texto in pedidos:
    pedido, mensagem = interpretar_pedido(texto)
    print(" ", texto)
    print(" ", mensagem, "→", pedido, "\n")


  Estou na Catedral da Sé e quero ir ao Terminal Rodoviário Jabaquara
  Interpretado via groq → {'origem': ('local', 'Catedral da Sé'), 'destino': ('local', 'Terminal Rodoviário Jabaquara'), 'acessibilidade': False} 

  to na se, bora pra pinacoteca, tô de cadeira de rodas
  Interpretado via groq → {'origem': ('estacao', 'Sé'), 'destino': ('local', 'Pinacoteca'), 'acessibilidade': True} 

  Preciso sair do Mosteiro de São Bento e chegar na São Judas
  Interpretado via groq → {'origem': ('local', 'Mosteiro de São Bento'), 'destino': ('estacao', 'São Judas'), 'acessibilidade': False} 

  Quero ir da Sé até a Avenida Paulista
  Não entendi origem/destino (resposta bruta: {'origem': 'Sé', 'destino': None, 'acessibilidade': False}) → None 



In [ ]:
def narrar_offline(r):
    if r["caminho"] is None:
        return (f"Não existe rota de {r['origem']} até {r['destino']} "
           f"com as estações bloqueadas: {', '.join(r['bloqueadas'])}.")
    texto = (f"Embarque em {r['origem']} e siga pela Linha 1-Azul até {r['destino']}: "
      f"{r['paradas']} parada(s), cerca de {r['tempo_min']} minutos.")
    if r["alertas"]:
        texto += " Atenção: " + "; ".join(r["alertas"]) + " (elevador em manutenção)."
    return texto

PROMPT_NARRADOR = """Você é o NARRADOR do MetrôBot SP. Explique a rota ao passageiro
em português, em no máximo 4 frases curtas e simpáticas.
Use SOMENTE os dados do JSON. Não invente horários, linhas, estações
ou atrações. Se "caminho" for null, explique que não há rota e cite as
estações bloqueadas. Se houver "alertas", destaque-os."""

def narrar(resultado):
    """Transforma o resultado da busca em explicação amigável."""
    dados = {k: resultado[k] for k in
    ("origem", "destino", "caminho", "paradas", "tempo_min", "bloqueadas", "alertas")}
    if PROVEDOR == "offline":
       return narrar_offline(resultado)
    try:
         return chamar_llm([{"role": "system", "content": PROMPT_NARRADOR},
    {"role": "user", "content": json.dumps(dados, ensure_ascii=False)}])
    except Exception as erro:
      return narrar_offline(resultado) + f" (narrador offline: {erro})"

In [ ]:
r = planejar({"origem": ("local", "Catedral da Sé"),
            "destino": ("local", "Pinacoteca"),
            "acessibilidade": True}, manutencao=["Luz"])
print(narrar(r))


Saia da estação Sé, siga para São Bento e depois para Luz.  
São 2 paradas e o percurso leva 4 minutos.  
Atenção: há alerta para o destino Luz.  
Não há estações bloqueadas.


In [ ]:
def desenhar_linha(resultado):
    caminho = set(resultado["caminho"] or [])
    visitados = set(resultado["visitados"])
    bloqueadas = set(resultado["bloqueadas"])
    linhas_html = []
    for estacao in LINHA_1_AZUL:
        if estacao in bloqueadas:
         cor, marca = "#d32f2f", " bloqueada"
        elif estacao in (resultado["origem"], resultado["destino"]) and estacao in caminho:
            cor, marca = "#0d47a1", " " + ("origem" if estacao == resultado["origem"] else "destino")
        elif estacao in caminho:
            cor, marca = "#1e88e5", "rota"
        elif estacao in visitados:
            cor, marca = "#9e9e9e", "visitada pela busca"
        else:
           cor, marca = "#e0e0e0", ""
        linhas_html.append(
    f"<div style='display:flex;align-items:center;gap:8px;font-family:sans-serif;font-size:13px'>"
    f"<span style='display:inline-block;width:14px;height:14px;border-radius:50%;background:{cor}'></span>"
    f"<span style='min-width:150px'>{estacao}</span><span style='color:#666'>{marca}</span></div>")
    return "<div style='border-left:4px solid #1e88e5;padding-left:8px'>" + "".join(linhas_html) + "</div>"

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
opcoes = ([(f" {local}", ("local", local)) for local in LOCAIS] +
          [(f" {estacao}", ("estacao", estacao)) for estacao in LINHA_1_AZUL])
txt_pedido = widgets.Textarea(placeholder="Ex.: Estou na Catedral da Sé e quero ir ao Terminal Rodoviário Jabaquara",
layout=widgets.Layout(width="95%", height="60px"))
btn_interpretar = widgets.Button(description=" Interpretar pedido", button_style="info")
dd_origem = widgets.Dropdown(options=opcoes, description="Origem:")
dd_destino = widgets.Dropdown(options=opcoes, value=("local", "Terminal Rodoviário Jabaquara"), description
="Destino:")
chk_acess = widgets.Checkbox(description="Preciso de acessibilidade")
sel_fechadas = widgets.SelectMultiple(options=LINHA_1_AZUL, description="Fechadas:", rows=5)
sel_manut = widgets.SelectMultiple(options=LINHA_1_AZUL, description="Elevador :", rows=5)
rb_algoritmo = widgets.RadioButtons(options=["BFS", "DFS"], description="Busca:")
btn_buscar = widgets.Button(description=" Buscar rota", button_style="success")
saida = widgets.Output()
def ao_interpretar(_):
    with saida:
      clear_output()
      pedido, msg = interpretar_pedido(txt_pedido.value)
      print(msg)
      if pedido:
          dd_origem.value = pedido["origem"]
          dd_destino.value = pedido["destino"]
          chk_acess.value = pedido["acessibilidade"]
          print(" Campos preenchidos. Confira e clique em 'Buscar rota'.")
def ao_buscar(_):
  with saida:
      clear_output()
      pedido = {"origem": dd_origem.value, "destino": dd_destino.value,
      "acessibilidade": chk_acess.value}
      r = planejar(pedido, sel_fechadas.value, sel_manut.value, rb_algoritmo.value)
      display(HTML(f"<h4>{r['algoritmo']}: {r['origem']} → {r['destino']}</h4>"))
      print(" ", narrar(r))
      print(f" Estações visitadas pela busca: {len(r['visitados'])}")
      print(f" Regras disparadas: {', '.join(r['regras_usadas'])}")
      display(HTML(desenhar_linha(r)))

btn_interpretar.on_click(ao_interpretar)
btn_buscar.on_click(ao_buscar)

painel = widgets.VBox([
    widgets.HTML("<h3> MetrôBot SP — Linha 1-Azul</h3>"),
    txt_pedido, btn_interpretar,
    widgets.HBox([dd_origem, dd_destino]),
    widgets.HBox([chk_acess, rb_algoritmo]),
    widgets.HBox([sel_fechadas, sel_manut]),
    btn_buscar, saida,

  ])



In [ ]:
display(painel)


In [ ]:
def rodar_testes():
    # 1. A Linha 1 tem 23 estações e as pontas têm só 1 vizinho
    assert len(GRAFO) == 23
    assert len(GRAFO["Tucuruvi"]) == 1 and len(GRAFO["Jabaquara"]) == 1

    # 2. BFS e DFS encontram o mesmo caminho numa linha reta
    c_bfs, _ = bfs(GRAFO, "Sé", "Vergueiro")
    c_dfs, _ = dfs(GRAFO, "Sé", "Vergueiro")
    assert c_bfs == c_dfs == ["Sé", "Japão-Liberdade", "São Joaquim", "Vergueiro"]

    # 3. Estação fechada no meio do caminho = sem rota
    caminho, _ = bfs(GRAFO, "Sé", "Jabaquara", bloqueadas={"Paraíso"})
    assert caminho is None

    # 4. Regra R2: local conhecido vira estação de destino
    r = planejar({"origem": ("local", "Catedral da Sé"),
    "destino": ("local", "Pinacoteca")})
    assert r["destino"] == "Luz" and r["paradas"] == 2

    # 5. Regras R4 + R5: acessibilidade + elevador em manutenção = alerta
    r = planejar({"origem": ("estacao", "Sé"), "destino": ("estacao", "Luz"),
    "acessibilidade": True}, manutencao=["Luz"])

    assert "destino: Luz" in r["alertas"]
    # 6. Passar POR uma estação sem elevador não gera alerta
    r = planejar({"origem": ("estacao", "Sé"), "destino": ("estacao", "Tiradentes"),
    "acessibilidade": True}, manutencao=["Luz"])
    assert r["alertas"] == [] and "Luz" in r["caminho"]
    print(" Todos os 6 testes passaram!")
rodar_testes()


 Todos os 6 testes passaram!


In [ ]:
%pip install networkx matplotlib

In [ ]:
#===== DESAFIO: MetrôBot SP 2.0 (COM TODOS OS BÔNUS) =====
from collections import deque
import heapq
import random
import networkx as nx
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# R1 - Modelagem dos Dados (Linhas e Cores)
LINHAS = {
    "Linha 1-Azul": [
        "Tucuruvi", "Parada Inglesa", "Jardim São Paulo", "Santana",
        "Carandiru", "Portuguesa-Tietê", "Armênia", "Tiradentes", "Luz",
        "São Bento", "Sé", "Japão-Liberdade", "São Joaquim", "Vergueiro",
        "Paraíso", "Ana Rosa", "Vila Mariana", "Santa Cruz",
        "Praça da Árvore", "Saúde", "São Judas", "Conceição", "Jabaquara",
    ],
    "Linha 2-Verde": [
        "Vila Madalena", "Sumaré", "Clínicas", "Consolação", "Trianon-Masp",
        "Brigadeiro", "Paraíso", "Ana Rosa", "Chácara Klabin",
        "Santos-Imigrantes", "Alto do Ipiranga", "Sacomã", "Tamanduateí",
        "Vila Prudente",
    ],
    "Linha 3-Vermelha": [
        "Palmeiras-Barra Funda", "Marechal Deodoro", "Santa Cecília",
        "República", "Anhangabaú", "Sé", "Pedro II", "Brás",
        "Bresser-Mooca", "Belém", "Tatuapé", "Carrão", "Penha",
        "Vila Matilde", "Guilhermina-Esperança", "Patriarca-Vila Ré",
        "Artur Alvim", "Corinthians-Itaquera",
    ]
}

CORES = {
    "Linha 1-Azul": "#1e88e5",
    "Linha 2-Verde": "#2e7d32",
    "Linha 3-Vermelha": "#d32f2f"
}

def construir_grafo_multilinhas(linhas):
    grafo = {}
    linhas_do_trecho = {}
    for nome_linha, estacoes in linhas.items():
        for i in range(len(estacoes) - 1):
            a, b = estacoes[i], estacoes[i + 1]
            if a not in grafo: grafo[a] = []
            if b not in grafo: grafo[b] = []
            if b not in grafo[a]: grafo[a].append(b)
            if a not in grafo[b]: grafo[b].append(a)
            if (a, b) not in linhas_do_trecho: linhas_do_trecho[(a, b)] = set()
            if (b, a) not in linhas_do_trecho: linhas_do_trecho[(b, a)] = set()
            linhas_do_trecho[(a, b)].add(nome_linha)
            linhas_do_trecho[(b, a)].add(nome_linha)
    return grafo, linhas_do_trecho

GRAFO_MULTI, LINHAS_DO_TRECHO = construir_grafo_multilinhas(LINHAS)
TODAS_ESTACOES = list(GRAFO_MULTI.keys())

def contar_baldeacoes(caminho, linhas_do_trecho):
    if not caminho or len(caminho) < 2: return 0, []
    baldeacoes = []
    opcoes_iniciais = list(linhas_do_trecho[(caminho[0], caminho[1])])
    if len(caminho) > 2:
        opcoes_prox = linhas_do_trecho[(caminho[1], caminho[2])]
        interseccao = set(opcoes_iniciais).intersection(opcoes_prox)
        linha_atual = list(interseccao)[0] if interseccao else opcoes_iniciais[0]
    else:
        linha_atual = opcoes_iniciais[0]

    for i in range(1, len(caminho) - 1):
        estacao_atual, proxima_estacao = caminho[i], caminho[i + 1]
        opcoes_prox = linhas_do_trecho[(estacao_atual, proxima_estacao)]
        if linha_atual not in opcoes_prox:
            nova_linha = list(opcoes_prox)[0]
            if i + 2 < len(caminho):
                opcoes_futuras = linhas_do_trecho[(proxima_estacao, caminho[i + 2])]
                inter = set(opcoes_prox).intersection(opcoes_futuras)
                if inter: nova_linha = list(inter)[0]
            baldeacoes.append((estacao_atual, nova_linha))
            linha_atual = nova_linha
    return len(baldeacoes), baldeacoes

LOCAIS_MULTI = {
    "Shopping Metrô Tucuruvi": "Tucuruvi", "Pinacoteca": "Luz", "Catedral da Sé": "Sé",
    "MASP": "Trianon-Masp", "Hospital das Clínicas": "Clínicas", "Aquário de SP": "Santos-Imigrantes",
    "Theatro Municipal": "Anhangabaú", "Neo Química Arena": "Corinthians-Itaquera", "Mercadão": "São Bento"
}

def fatos_base_multi():
    fatos = set()
    for linha, estacoes in LINHAS.items():
        for estacao in estacoes:
            fatos.add(("estacao", estacao))
            fatos.add(("pertence", estacao, linha))
    for local, estacao in LOCAIS_MULTI.items():
        fatos.add(("proximo_de", local, estacao))
    return fatos

def consultar_multi(fatos, predicado):
    return [f[1:] for f in fatos if f[0] == predicado]

# BÔNUS 3: Regra de Linha Paralisada -> Deduz Bloqueio Automático
def r_linha_paralisada(fatos):
    novos = set()
    paralisadas = consultar_multi(fatos, "linha_paralisada")
    pertences = consultar_multi(fatos, "pertence")
    for (l_paral,) in paralisadas:
        for (e, l_pert) in pertences:
            if l_paral == l_pert:
                novos.add(("bloqueada", e)) # Bloqueia todas as estações da linha paralisada
                novos.add(("alerta", "linha_paralisada", e))
    return novos

def r_integracao(fatos):
    novos = set()
    pertences = consultar_multi(fatos, "pertence")
    estacao_linhas = {}
    for (e, l) in pertences:
        if e not in estacao_linhas: estacao_linhas[e] = set()
        estacao_linhas[e].add(l)
    for e, linhas in estacao_linhas.items():
        if len(linhas) > 1: novos.add(("integracao", e))
    return novos

def r_origem(fatos):
    novos = set()
    for (local,) in consultar_multi(fatos, "usuario_esta_em"):
        for (l, e) in consultar_multi(fatos, "proximo_de"):
            if l == local: novos.add(("origem", e))
    for (e,) in consultar_multi(fatos, "usuario_esta_na_estacao"): novos.add(("origem", e))
    return novos

def r_destino(fatos):
    novos = set()
    for (local,) in consultar_multi(fatos, "usuario_quer_ir"):
        for (l, e) in consultar_multi(fatos, "proximo_de"):
            if l == local: novos.add(("destino", e))
    for (e,) in consultar_multi(fatos, "usuario_quer_ir_estacao"): novos.add(("destino", e))
    return novos

def r_bloqueio(fatos):
    return {("bloqueada", e) for (e,) in consultar_multi(fatos, "fechada")}

def r_acessibilidade(fatos):
    if not consultar_multi(fatos, "precisa_acessibilidade"): return set()
    return {("inacessivel", e) for (e,) in consultar_multi(fatos, "elevador_em_manutencao")}

def r_alerta(fatos):
    novos = set()
    inacessiveis = {e for (e,) in consultar_multi(fatos, "inacessivel")}
    for papel in ("origem", "destino"):
        for (e,) in consultar_multi(fatos, papel):
            if e in inacessiveis: novos.add(("alerta", papel, e))
    return novos

REGRAS_MULTI = [
    ("R1 origem", "Vl Ve (usuario_esta_em(l) ^ proximo_de(l,e) -> origem(e))", r_origem),
    ("R2 destino", "Vl Ve (usuario_quer_ir(l) ^ proximo_de(l,e) -> destino(e))", r_destino),
    ("R3 bloqueio", "Ve (fechada(e) -> bloqueada(e))", r_bloqueio),
    ("R4 acessibilidade", "Ve (precisa_acessibilidade ^ elevador_em_manutencao(e) -> inacessivel(e))", r_acessibilidade),
    ("R5 alerta", "Vp Ve (papel(p,e) ^ inacessivel(e) -> alerta(p,e))", r_alerta),
    ("R6 integracao", "Ve Vl1 Vl2 (pertence(e, l1) ^ pertence(e, l2) ^ l1!=l2 -> integracao(e))", r_integracao),
    ("R7 linha_paralisada", "Vl Ve (linha_paralisada(l) ^ pertence(e, l) -> bloqueada(e))", r_linha_paralisada)
]

def encadear_para_frente_multi(fatos, regras):
    fatos = set(fatos)
    justificativas = {}
    while True:
        novos_na_rodada = set()
        for nome, formula, regra in regras:
            for fato in regra(fatos) - fatos:
                novos_na_rodada.add(fato)
                justificativas[fato] = nome
        if not novos_na_rodada:
            return fatos, justificativas
        fatos |= novos_na_rodada

# --- ALGORITMOS DE BUSCA ---
def bfs_multi(grafo, origem, destino, bloqueadas=()):
    if origem in bloqueadas or destino in bloqueadas: return None, []
    fila = deque([origem])
    pai = {origem: None}
    ordem_visita = []
    while fila:
        atual = fila.popleft()
        ordem_visita.append(atual)
        if atual == destino:
            c = []
            while atual is not None:
                c.append(atual)
                atual = pai.get(atual)
            return list(reversed(c)), ordem_visita
        for vizinho in grafo.get(atual, []):
            if vizinho not in pai and vizinho not in bloqueadas:
                pai[vizinho] = atual
                fila.append(vizinho)
    return None, ordem_visita

def dfs_multi(grafo, origem, destino, bloqueadas=()):
    if origem in bloqueadas or destino in bloqueadas: return None, []
    visitados = set()
    ordem_visita = []
    def explorar(atual, caminho):
        visitados.add(atual)
        ordem_visita.append(atual)
        if atual == destino: return caminho
        for vizinho in grafo.get(atual, []):
            if vizinho not in visitados and vizinho not in bloqueadas:
                resultado = explorar(vizinho, caminho + [vizinho])
                if resultado: return resultado
        return None
    return explorar(origem, [origem]), ordem_visita

# BÔNUS 1 E 2: BUSCA COM DIJKSTRA E HEAPQ (Menos Baldeações / Tempo Real)
def busca_dijkstra_ou_baldeacao(grafo, linhas_do_trecho, origem, destino, bloqueadas, modo):
    if origem in bloqueadas or destino in bloqueadas: return None, []
    pq = []
    visitados_estado = {} # O estado passa a ser (estacao, linha_atual)
    ordem_visita = []
    visitados_esforco = set()

    linhas_origem = set()
    for v in grafo.get(origem, []):
        if v not in bloqueadas:
            linhas_origem.update(linhas_do_trecho.get((origem, v), set()))

    if not linhas_origem: linhas_origem = {None}
    for l in linhas_origem:
        heapq.heappush(pq, (0, 0, origem, l, [origem]))
        visitados_estado[(origem, l)] = (0, 0)

    while pq:
        custo_prio, paradas, atual, linha_atual, caminho = heapq.heappop(pq)
        if atual not in visitados_esforco:
            visitados_esforco.add(atual)
            ordem_visita.append(atual)

        if atual == destino:
            return caminho, ordem_visita

        for vizinho in grafo.get(atual, []):
            if vizinho in bloqueadas: continue

            opcoes_linha = linhas_do_trecho.get((atual, vizinho), set())
            for nova_linha in opcoes_linha:
                trocou_linha = (nova_linha != linha_atual) and (linha_atual is not None)

                if modo == "DIJKSTRA":
                    # Tempo real: +2 por trecho, +5 por baldeacao
                    novo_custo_prio = custo_prio + 2 + (5 if trocou_linha else 0)
                else:
                    # MENOS_BALDEACOES: Foca em baldeacoes primeiro (peso alto), depois paradas
                    novo_custo_prio = custo_prio + (100 if trocou_linha else 1)

                nova_parada = paradas + 1
                novo_estado = (vizinho, nova_linha)

                if novo_estado not in visitados_estado or (novo_custo_prio, nova_parada) < visitados_estado[novo_estado]:
                    visitados_estado[novo_estado] = (novo_custo_prio, nova_parada)
                    heapq.heappush(pq, (novo_custo_prio, nova_parada, vizinho, nova_linha, caminho + [vizinho]))

    return None, ordem_visita

TEMPO_POR_TRECHO = 2
TEMPO_BALDEACAO = 5

def planejar_multi(pedido, fechadas=(), manutencao=(), linhas_paralisadas=(), algoritmo="BFS"):
    fatos = fatos_base_multi()
    tipo_o, nome_o = pedido["origem"]
    tipo_d, nome_d = pedido["destino"]

    fatos.add(("usuario_esta_em", nome_o) if tipo_o == "local" else ("usuario_esta_na_estacao", nome_o))
    fatos.add(("usuario_quer_ir", nome_d) if tipo_d == "local" else ("usuario_quer_ir_estacao", nome_d))

    if pedido.get("acessibilidade"): fatos.add(("precisa_acessibilidade",))
    for e in fechadas: fatos.add(("fechada", e))
    for e in manutencao: fatos.add(("elevador_em_manutencao", e))
    for l in linhas_paralisadas: fatos.add(("linha_paralisada", l))

    fatos, justificativas = encadear_para_frente_multi(fatos, REGRAS_MULTI)

    try:
        origem = consultar_multi(fatos, "origem")[0][0]
        destino = consultar_multi(fatos, "destino")[0][0]
    except IndexError:
        return {"erro": "Origem ou destino não identificados pela lógica."}

    bloqueadas = {e for (e,) in consultar_multi(fatos, "bloqueada")}
    alertas = consultar_multi(fatos, "alerta")

    if algoritmo in ["DIJKSTRA", "MENOS_BALDEACOES"]:
        caminho, visitados = busca_dijkstra_ou_baldeacao(GRAFO_MULTI, LINHAS_DO_TRECHO, origem, destino, bloqueadas, algoritmo)
    else:
        buscar = bfs_multi if algoritmo == "BFS" else dfs_multi
        caminho, visitados = buscar(GRAFO_MULTI, origem, destino, bloqueadas)

    qtd_baldeacoes, baldeacoes = 0, []
    tempo_total = None
    if caminho:
        qtd_baldeacoes, baldeacoes = contar_baldeacoes(caminho, LINHAS_DO_TRECHO)
        tempo_total = (len(caminho) - 1) * TEMPO_POR_TRECHO + (qtd_baldeacoes * TEMPO_BALDEACAO)

    return {
        "origem": origem, "destino": destino, "algoritmo": algoritmo,
        "caminho": caminho, "visitados": visitados, "bloqueadas": sorted(bloqueadas),
        "alertas": [f"{papel}: {e}" for papel, e in alertas],
        "paradas": len(caminho) - 1 if caminho else None,
        "baldeacoes": baldeacoes, "qtd_baldeacoes": qtd_baldeacoes,
        "tempo_min": tempo_total, "regras_usadas": sorted(set(justificativas.values()))
    }

def narrar_multi(r):
    if "erro" in r: return r["erro"]
    if r["caminho"] is None:
        return f"Não existe rota de {r['origem']} até {r['destino']} com os bloqueios atuais."
    texto = f"Embarque em {r['origem']} e siga até {r['destino']}: {r['paradas']} parada(s), cerca de {r['tempo_min']} minutos."
    if r['qtd_baldeacoes'] > 0:
        detalhes_bald = ", ".join([f"na estação {est} para a {lin}" for est, lin in r['baldeacoes']])
        texto += f" Você fará {r['qtd_baldeacoes']} baldeação(ões): {detalhes_bald}."
    if r["alertas"]:
        texto += " Atenção: " + "; ".join(r["alertas"]) + "."
    return texto

def desenhar_linhas_multi(resultado):
    if "erro" in resultado or not resultado["caminho"]:
        return "<div>Sem rota ou erro no planejamento.</div>"
    caminho = resultado["caminho"]
    linhas_html = []
    for i, estacao in enumerate(caminho):
        linha_cor = "#9e9e9e"
        if i < len(caminho) - 1:
            opcoes = LINHAS_DO_TRECHO[(caminho[i], caminho[i + 1])]
            linha_cor = CORES[list(opcoes)[0]]
        elif i > 0:
            opcoes = LINHAS_DO_TRECHO[(caminho[i - 1], caminho[i])]
            linha_cor = CORES[list(opcoes)[0]]

        marca = ""
        if estacao == resultado["origem"]: marca = "origem"
        elif estacao == resultado["destino"]: marca = "destino"
        elif any(est == estacao for est, lin in resultado.get("baldeacoes", [])): marca = "baldeação"

        linhas_html.append(
            f"<div style='display:flex;align-items:center;gap:8px;font-family:sans-serif;font-size:13px'>"
            f"<span style='display:inline-block;width:14px;height:14px;border-radius:50%;background:{linha_cor}'></span>"
            f"<span style='min-width:150px'>{estacao}</span><span style='color:#666'><b>{marca}</b></span></div>"
        )
    return f"<div style='padding-left: 8px'>{''.join(linhas_html)}</div>"

# BÔNUS 4: Mapa de Verdade (NetworkX + Matplotlib)
def desenhar_mapa_networkx():
    G = nx.Graph()
    for (u, v), linhas in LINHAS_DO_TRECHO.items():
        l = list(linhas)[0]
        G.add_edge(u, v, color=CORES[l])

    cores_edges = [G[u][v]['color'] for u,v in G.edges()]
    plt.figure(figsize=(12,8))
    pos = nx.spring_layout(G, seed=42)
    nx.draw(G, pos, with_labels=True, node_size=150, node_color='lightgray', edge_color=cores_edges, width=3, font_size=8)
    plt.title("Mapa de Verdade - Metrô SP (Linhas 1, 2 e 3)")
    plt.show()

# BÔNUS 5: Relatório de Esforço (BFS vs DFS)
def gerar_relatorio_esforco():
    viagens = []
    bfs_esf = []
    dfs_esf = []
    estacoes_validas = list(TODAS_ESTACOES)
    for _ in range(10):
        o, d = random.sample(estacoes_validas, 2)
        viagens.append(f"{o[:4]}... -> {d[:4]}...")
        _, v_bfs = bfs_multi(GRAFO_MULTI, o, d, [])
        _, v_dfs = dfs_multi(GRAFO_MULTI, o, d, [])
        bfs_esf.append(len(v_bfs))
        dfs_esf.append(len(v_dfs))

    x = range(10)
    plt.figure(figsize=(10,5))
    plt.bar([pos - 0.2 for pos in x], bfs_esf, width=0.4, label='BFS', color='#1e88e5')
    plt.bar([pos + 0.2 for pos in x], dfs_esf, width=0.4, label='DFS', color='#ff9800')
    plt.xticks(x, viagens, rotation=45, ha='right')
    plt.ylabel('Estações Visitadas')
    plt.legend()
    plt.title('Esforço de Busca: BFS vs DFS (10 viagens aleatórias)')
    plt.tight_layout()
    plt.show()


# TODO 9: Montar o novo Painel (App 2.0 Turbinado)
opcoes_multi = [(f"📍 (local) {local}", ("local", local)) for local in LOCAIS_MULTI] + \
               [(f"🚉 (estação) {estacao}", ("estacao", estacao)) for estacao in TODAS_ESTACOES]

dd_origem_multi = widgets.Dropdown(options=opcoes_multi, description="Origem:")
dd_destino_multi = widgets.Dropdown(options=opcoes_multi, description="Destino:")
chk_acess_multi = widgets.Checkbox(description="♿ Acessibilidade")

sel_fechadas_multi = widgets.SelectMultiple(options=TODAS_ESTACOES, description="Fechadas:", rows=6)
# Novo widget de Bônus 3
sel_linhas_paralisadas = widgets.SelectMultiple(options=list(LINHAS.keys()), description="🚫 Linha Paralisada:", rows=3)

# Novas opções de Bônus 1 e 2
rb_algoritmo_multi = widgets.RadioButtons(options=["BFS", "DFS", "DIJKSTRA (Tempo)", "MENOS_BALDEACOES"], description="Busca:")

btn_buscar_multi = widgets.Button(description="🔍 Buscar Rota", button_style="success")
btn_limpar_multi = widgets.Button(description="🧹 Limpar Fechadas/Paralisadas", button_style="warning")
btn_mapa = widgets.Button(description="🗺️ Gerar Mapa", button_style="info")
btn_relatorio = widgets.Button(description="📊 Relatório de Esforço", button_style="info")
saida_multi = widgets.Output()

def ao_limpar_multi(_):
    sel_fechadas_multi.value = ()
    sel_linhas_paralisadas.value = ()
    with saida_multi:
        clear_output()
        print("🧹 Restrições limpas! Você já pode rodar os cenários 'normais'.")

def ao_buscar_multi(_):
    with saida_multi:
        clear_output()
        pedido = {
            "origem": dd_origem_multi.value,
            "destino": dd_destino_multi.value,
            "acessibilidade": chk_acess_multi.value
        }
        alg = rb_algoritmo_multi.value.split(" ")[0] # Pega apenas a palavra chave
        r = planejar_multi(pedido, fechadas=sel_fechadas_multi.value, linhas_paralisadas=sel_linhas_paralisadas.value, algoritmo=alg)
        display(HTML(f"<h4>{r.get('algoritmo', 'Erro')}: {r.get('origem', '-')} ➡️ {r.get('destino', '-')}</h4>"))
        print("🗣️", narrar_multi(r))
        if r.get("caminho"):
            print(f"\n👣 Estações visitadas: {len(r['visitados'])}")
            print(f"📜 Regras disparadas: {', '.join(r.get('regras_usadas', []))}")
            display(HTML(desenhar_linhas_multi(r)))

def ao_clicar_mapa(_):
    with saida_multi:
        clear_output()
        desenhar_mapa_networkx()

def ao_clicar_relatorio(_):
    with saida_multi:
        clear_output()
        gerar_relatorio_esforco()

btn_buscar_multi.on_click(ao_buscar_multi)
btn_limpar_multi.on_click(ao_limpar_multi)
btn_mapa.on_click(ao_clicar_mapa)
btn_relatorio.on_click(ao_clicar_relatorio)

painel_multi = widgets.VBox([
    widgets.HTML("<h2>🚇 MetrôBot SP 2.0 (Com 5 Bônus)</h2>"),
    widgets.HBox([dd_origem_multi, dd_destino_multi]),
    widgets.HBox([chk_acess_multi, rb_algoritmo_multi]),
    widgets.HBox([sel_fechadas_multi, sel_linhas_paralisadas]),
    widgets.HBox([btn_buscar_multi, btn_limpar_multi, btn_mapa, btn_relatorio]),
    saida_multi
])
display(painel_multi)